In [46]:
%%bash
sudo apt-get update &> /dev/null
sudo apt-get install -y pciutils 1> /dev/null
curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...


>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [47]:
%%bash
pip install -q "polars[gpu]" \
    spacy \
    seaborn \
    textstat \
    ollama \
    matplotlib \
    pyarrow \
    typing-extensions \
    pyarrow==24.0.0 \
    seaborn==0.13.2 \
    spacy==3.8.13 \
    textstat==0.7.13 \
    transformers==5.10.2 \
    click==8.0.4 \
    click-completion>=0.5.2

spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 178.2 MB/s  0:00:00


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import os
import re
import json
import numpy
import polars
import ollama
import spacy
import seaborn
import textstat

from spacy.language import Language
from spacy.tokens import Doc
from pathlib import Path
from itertools import permutations
from matplotlib import pyplot
from capstone_lib import CapstoneLib 

BASE_PATH = "/workspace"
os.chdir(f"{BASE_PATH}/uol_capstone_project")


In [2]:
# 1  Cuad Data loading
PATH_CUAD_PARQUET = Path(BASE_PATH).rglob("**/*/0001_cuad_dataset.parquet")
PATH_CUAD_JSON = Path(BASE_PATH).rglob("**/*/CUADv1.json")

CUAD_PARQUET = next(PATH_CUAD_PARQUET, None)
CUAD_JSON = next(PATH_CUAD_JSON, None)


if os.path.exists(CUAD_PARQUET):
    cuad_legal = CapstoneLib.load_parquet(CUAD_PARQUET)

    with open(CUAD_JSON) as cuad:
        cuad_json = json.load(cuad.buffer)

    legal_pieces = {}

    for data in cuad_json.get("data"):
        full_context = ""
        for paragraph in data.get("paragraphs"):
            if context := paragraph.get("context", ""):
                full_context += context
        legal_pieces[data.get("title")] = full_context

    cuad_legal = polars.DataFrame(legal_pieces)
    cuad_legal = cuad_legal.transpose(include_header=True)
    cuad_legal.columns = ["title", "paragraphs"]

    cuad_legal.write_parquet("parquet_files/0001_cuad_dataset.parquet")


In [3]:
# 2 Cuad data parsing

# Here we are trying to load cached file if it exists, otherwise we will need to reprocess everything
PATH_EXTRACTED_SENTENCES = Path(".").rglob("**/*/0002_cuad_dataset.parquet")
CUAD_EXTRACTED = next(PATH_EXTRACTED_SENTENCES, "non_existent.parquet")

nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")
nlp.add_pipe("parser")
nlp.initialize()

if os.path.exists(CUAD_EXTRACTED):
    cuad_legal = polars.read_parquet(CUAD_EXTRACTED)
else:
    cuad_legal = cuad_legal.with_columns(
        polars.col("paragraphs").map_elements(
            lambda paragraph: [p.text for p in nlp(paragraph).sents],
            return_dtype=polars.List(polars.String)
        ).alias("sentences")
    )
    cuad_legal = (
        cuad_legal
            .explode("sentences", empty_as_null=False)
            .filter(
                polars.col("sentences").is_not_null() & 
                (polars.col("sentences").str.len_chars() > 0)
            )
            .with_columns(
                polars.col("sentences").map_elements(
                    lambda text: re.sub(r"\s{2,}"," ", text)
                )
            )
            .with_columns(
                polars.col("sentences").str.strip_chars()
            )
    )
    cuad_legal.write_parquet("parquet_files/0002_cuad_dataset.parquet")

In [4]:
# 3 Applying metrics

cuad_legal = CapstoneLib.apply_metrics(cuad_legal)

cuad_legal

title,paragraphs,sentences,fre,fkgl,fog,dale_chall,lix,complexity_score
str,str,str,f64,f64,f64,f64,f64,f64
"""LIMEENERGYCO_09_09_1999-EX-10-…","""EXHIBIT 10.6 …","""EXHIBIT 10.6 DISTRIBUTOR AGREE…",17.891765,14.987059,18.564706,11.445876,61.117647,24.801411
"""LIMEENERGYCO_09_09_1999-EX-10-…","""EXHIBIT 10.6 …","""RECITALS A. The Company's Busi…",32.56,9.96,18.0,10.2005,65.0,27.1441
"""LIMEENERGYCO_09_09_1999-EX-10-…","""EXHIBIT 10.6 …","""The Company is presently engag…",26.281429,18.288571,19.714286,11.237357,72.142857,29.5329
"""LIMEENERGYCO_09_09_1999-EX-10-…","""EXHIBIT 10.6 …","""The Company may engage in the …",22.085455,18.37697,19.260606,11.972088,72.393939,28.817812
"""LIMEENERGYCO_09_09_1999-EX-10-…","""EXHIBIT 10.6 …","""B. Representations.""",-48.995,20.59,20.8,19.5257,52.0,12.78414
…,…,…,…,…,…,…,…,…
"""PerformanceSportsBrandsInc_201…","""ENDORSEMENT AGREEMENT enter…","""This Agreement may be executed…",46.665,14.203333,20.0,8.808833,53.333333,28.6021
"""PerformanceSportsBrandsInc_201…","""ENDORSEMENT AGREEMENT enter…","""Page 11 of 12 Source: PERFORMA…",94.3,2.342222,8.044444,16.364011,20.111111,28.232358
"""PerformanceSportsBrandsInc_201…","""ENDORSEMENT AGREEMENT enter…","""S-1, 9/9/2011 IN WITNESS WHERE…",57.605,10.69,12.436364,11.187245,49.272727,28.238267
